# Introducción

El presente análisis tiene como objetivo evaluar los resultados de la prueba A/B **recommender_system_test**, llevada a cabo por una tienda en línea internacional entre el 7 y el 21 de diciembre de 2020.

La prueba busca determinar si la introducción de un sistema de recomendaciones mejorado genera un incremento de al menos 10% en la conversión de usuarios en cada etapa del embudo: vistas de página de producto (`product_page`), adición al carrito (`product_card`) y compras (`purchase`).

Antes de analizar los resultados, se verificará que la prueba fue ejecutada correctamente, revisando posibles contaminaciones de muestra, solapamiento con eventos de marketing y distribución equitativa entre grupos.

# Preparación de los datos 

Comenzamos importando nuestras librerías y cargando los DataSets

In [100]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as st
from statsmodels.stats.proportion import proportions_ztest

In [101]:
marketing_events = pd.read_csv("datasets/ab_project_marketing_events_us.csv")
ab_events = pd.read_csv("datasets/final_ab_events_upd_us.csv")
new_users = pd.read_csv("datasets/final_ab_new_users_upd_us.csv")
participants = pd.read_csv("datasets/final_ab_participants_upd_us.csv")

# Exploración de los datos

A continuación utilizaremos los métodos `.info()`, `.head()` y `.tail()` para obtener información inicial sobre nuestros DataFrames y buscaremos valores duplicados y nulos

## marketing_events

In [102]:
marketing_events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   name       14 non-null     object
 1   regions    14 non-null     object
 2   start_dt   14 non-null     object
 3   finish_dt  14 non-null     object
dtypes: object(4)
memory usage: 580.0+ bytes


In [103]:
marketing_events.head()

,name,regions,start_dt,finish_dt
0,Christmas&New Year Promo,"EU, N.America",2020-12-25,2021-01-03
1,St. Valentine's Day Giveaway,"EU, CIS, APAC, N.America",2020-02-14,2020-02-16
2,St. Patric's Day Promo,"EU, N.America",2020-03-17,2020-03-19
3,Easter Promo,"EU, CIS, APAC, N.America",2020-04-12,2020-04-19
4,4th of July Promo,N.America,2020-07-04,2020-07-11


In [104]:
marketing_events.tail()

,name,regions,start_dt,finish_dt
9,Victory Day CIS (May 9th) Event,CIS,2020-05-09,2020-05-11
10,CIS New Year Gift Lottery,CIS,2020-12-30,2021-01-07
11,Dragon Boat Festival Giveaway,APAC,2020-06-25,2020-07-01
12,Single's Day Gift Promo,APAC,2020-11-11,2020-11-12
13,Chinese Moon Festival,APAC,2020-10-01,2020-10-07


In [105]:
marketing_events.duplicated().sum()

np.int64(0)

El dataset `marketing_events` consta de cuatro columnas y 14 filas. Contiene el calendario de eventos de marketing para 2020

No hay valores nulos ni duplicados, hace falta cambiar el tipo de datos de las columnas `start_dt` y `finish_dt` a datetime

## ab_events

In [106]:
ab_events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 423761 entries, 0 to 423760
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   user_id     423761 non-null  object 
 1   event_dt    423761 non-null  object 
 2   event_name  423761 non-null  object 
 3   details     60314 non-null   float64
dtypes: float64(1), object(3)
memory usage: 12.9+ MB


In [107]:
ab_events.head()

,user_id,event_dt,event_name,details
0,E1BDDCE0DAFA2679,2020-12-07 20:22:03,purchase,99.99
1,7B6452F081F49504,2020-12-07 09:22:53,purchase,9.99
2,9CD9F34546DF254C,2020-12-07 12:59:29,purchase,4.99
3,96F27A054B191457,2020-12-07 04:02:40,purchase,4.99
4,1FD7660FDF94CA1F,2020-12-07 10:15:09,purchase,4.99


In [108]:
ab_events.tail()

,user_id,event_dt,event_name,details
423756,245E85F65C358E08,2020-12-30 19:35:55,login,NaN
423757,9385A108F5A0A7A7,2020-12-30 10:54:15,login,NaN
423758,DB650B7559AC6EAC,2020-12-30 10:59:09,login,NaN
423759,F80C9BDDEA02E53C,2020-12-30 09:53:39,login,NaN
423760,7AEC61159B672CC5,2020-12-30 11:36:13,login,NaN


In [109]:
ab_events.duplicated().sum()

np.int64(0)

El dataset `ab_events` contiene cuatro columnas y 423761 filas. Contiene información sobre los eventos realizados por los usuarios dentro de la tienda.

No cuenta con valores nulos a excepción de la columna `details` lo que es de esperarse debido a que solo los eventos *purchase* registran información (el tamaño de compra). Se conservaran para realizar cálculos sobre el tamaño de las compras. 

Tampoco hay valores duplicados.

Se debe cambiar el tipo de datos de la columna `event_dt` a datetime

## new_users

In [110]:
new_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58703 entries, 0 to 58702
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     58703 non-null  object
 1   first_date  58703 non-null  object
 2   region      58703 non-null  object
 3   device      58703 non-null  object
dtypes: object(4)
memory usage: 1.8+ MB


In [111]:
new_users.head()

,user_id,first_date,region,device
0,D72A72121175D8BE,2020-12-07,EU,PC
1,F1C668619DFE6E65,2020-12-07,N.America,Android
2,2E1BF1D4C37EA01F,2020-12-07,EU,PC
3,50734A22C0C63768,2020-12-07,EU,iPhone
4,E1BDDCE0DAFA2679,2020-12-07,N.America,iPhone


In [112]:
new_users.tail()

,user_id,first_date,region,device
58698,1DB53B933257165D,2020-12-20,EU,Android
58699,538643EB4527ED03,2020-12-20,EU,Mac
58700,7ADEE837D5D8CBBD,2020-12-20,EU,PC
58701,1C7D23927835213F,2020-12-20,EU,iPhone
58702,8F04273BB2860229,2020-12-20,EU,Android


In [113]:
new_users.duplicated().sum()

np.int64(0)

Verificamos la existencia de duplicados implícitos

In [114]:
new_users["user_id"].duplicated().sum()

np.int64(0)

El DataFrame `new_users` contiene cuatro columnas y 58703 filas. Contiene información sobre la ubicación, dispositivo y fecha de inscripción de cada usuario.

No hay valores nulos ni duplicados implícitos o explícitos

El tipo de datos de la columna `first_date` debe cambiarse a datetime

## participants

In [115]:
participants.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
dtypes: object(3)
memory usage: 340.6+ KB


In [116]:
participants.head()

,user_id,group,ab_test
0,D1ABA3E2887B6A73,A,recommender_system_test
1,A7A3664BD6242119,A,recommender_system_test
2,DABC14FDDFADD29E,A,recommender_system_test
3,04988C5DF189632E,A,recommender_system_test
4,4FF2998A348C484F,A,recommender_system_test


In [117]:
participants.tail()

,user_id,group,ab_test
14520,1D302F8688B91781,B,interface_eu_test
14521,3DE51B726983B657,A,interface_eu_test
14522,F501F79D332BE86C,A,interface_eu_test
14523,63FBE257B05F2245,A,interface_eu_test
14524,79F9ABFB029CF724,B,interface_eu_test


In [118]:
participants.duplicated().sum()

np.int64(0)

Verificamos si existen usuarios registrados en ambos grupos

In [119]:
users_in_both_groups = participants.groupby("user_id")["group"].nunique()

users_in_both_groups = users_in_both_groups[users_in_both_groups > 1]

print(f"Hay {len(users_in_both_groups)} usuarios en ambos grupos")

Hay 441 usuarios en ambos grupos


El DataFrame `participants` contiene tres columnas y 14525 filas. Contiene información sobre el grupo y prueba a la que pertenece cada participante. 

En vista de que el presente análisis esta enfocado en la prueba **recommender_system_test** filtraremos nuestro DataFrame para conservar unicamente a los participantes de esa prueba

No cuenta con valores nulos ni duplicados explícitos sin embargo existen 441 registros de participantes en ambos grupos. Estos podrían ser participantes registrados en mas de una prueba A/B, lo verificaremos después de filtrar las pruebas. 

# Procesamiento de los datos

Definimos una función para cambiar el tipo de datos de las columnas referentes a fechas de cada DataFrame

In [120]:
def to_date(df, col):
    df[col] = pd.to_datetime(df[col])
    print(df[col].dtype)

## marketing_events

Cambiamos el tipo de datos de las columnas `start_dt` y `finish_dt`

In [121]:
to_date(events, "start_dt")
to_date(events, "finish_dt")

datetime64[ns]
datetime64[ns]


## ab_events

Convertimos el tipo de datos de la columna `event_dt` 

In [122]:
to_date(ab_events, "event_dt")

datetime64[ns]


## new_users

Convertimos el tipo de datos de la columna `first_date`

In [126]:
to_date(new_users, "first_date")

datetime64[ns]


## participants

Filtramos a los usuarios que no participan en la prueba **recommender_system_test**

In [123]:
participants = participants[participants["ab_test"] == "recommender_system_test"]

print(f"Total participantes: {len(participants)}")

Total participantes: 3675


Verificamos si existen usuarios registrados en ambos grupos

In [124]:
users_in_both_groups = participants.groupby("user_id")["group"].nunique()

users_in_both_groups = users_in_both_groups[users_in_both_groups > 1]

print(f"Hay {len(users_in_both_groups)} usuarios en ambos grupos")

Hay 0 usuarios en ambos grupos


Revisamos la cantidad de usuarios en cada grupo

In [125]:
users_by_group = participants.groupby("group")["user_id"].nunique()

print(users_by_group)

group
A    2747
B     928
Name: user_id, dtype: int64


Los grupos están muy desbalanceados lo que podria deberse a que hay usuarios de otras regiones y periodos registrados, para filtrarlos comenzaremos fusionando nuestros DataFrames `participants` y `new_users`

In [132]:
ab_data = participants.merge(new_users, on="user_id", how="left")

print(len(ab_data))

3675


A continuación, filtramos a los participantes que no pertenezcan a **EU**

In [138]:
ab_data = ab_data[ab_data["region"] == "EU"]

Verificamos la cantidad de participantes por grupo

In [135]:
users_by_group_clean = ab_data.groupby("group")["user_id"].nunique()

print(users_by_group_clean)

group
A    2747
B     928
Name: user_id, dtype: int64


Verificamos que los usuarios correspondan al periodo de la prueba

In [136]:
print(ab_data["first_date"].min())
print(ab_data["first_date"].max())


2020-12-07 00:00:00
2020-12-21 00:00:00


Tras filtrar por región (EU) y verificar el período de registro, el desbalance entre grupos persiste: grupo A con 2,747 usuarios y grupo B con 928. Este desbalance no tiene una explicación clara en los datos disponibles y representa una limitación importante de la prueba. Los resultados deben interpretarse con cautela ya que la diferencia en tamaño muestral puede influir en la significancia estadística.

# Análisis exploratorio de los datos (EDA)

## ¿El número de eventos por usuario está distribuido equitativamente entre grupos?